# CoursIA pour les agents — 02 · Appeler les interfaces agentiques

Part of #27071

Le carnet 01 (inventaire) a **mesuré** ce que le dépôt offre aux agents : 1 serveur MCP, 622 outils CLI, 1 bridge notebook → service HTTP. Ce carnet répond à la question suivante : **comment un agent les appelle-t-il concrètement ?**

Trois contrats d'appel, un par classe :

| Classe | Contrat d'appel | Ce que le carnet démontre |
|---|---|---|
| **CLI** (`scripts/`) | `python <script> [args]` | les outils répondent à `--help` (sortie réelle) |
| **MCP** (`.mcp.json`) | serveur **stdio** : `command` + `args` | ligne de démarrage réelle + binaire présent |
| **Bridge** | service **HTTP** : `--host` / `--port` | l'interface réseau qu'un notebook interroge |

Chaque contrat est **démontré par exécution réelle**, pas décrit.

## Méthode — l'interface s'exécute, on ne l'affirme pas

Même règle que le carnet 01 : chaque affirmation du carnet est **produite par son code** et **re-exécutée** (règle C.2). « L'outil est invocable » veut dire que `subprocess` a réellement appelé l'outil et capté sa sortie ; « le binaire MCP existe » veut dire que le code a réellement vérifié le chemin. La racine du dépôt est **recherchée** (présence de `scripts/` + `MyIA.AI.Notebooks/`), jamais hardcodée.

In [1]:
import os
import subprocess
import sys
from pathlib import Path


def find_root(start: Path) -> Path:
    """Remonter jusqu'a la racine du depot (scripts/ + MyIA.AI.Notebooks/)."""
    p = start.resolve()
    for _ in range(8):
        if (p / "scripts").is_dir() and (p / "MyIA.AI.Notebooks").is_dir():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise RuntimeError("racine du depot introuvable")


ROOT = find_root(Path.cwd())
print("Racine du depot :", ROOT)

# Force l'enfant a emettre du UTF-8 (sinon la console cp1252 du siege
# corrompt les accents) ; le parent decode lui aussi en UTF-8.
UTF8_CHILD = {**os.environ, "PYTHONIOENCODING": "utf-8"}

# --- Classe 1 : outils CLI (echantillon des 622 du carnet 01) ---
cli_samples = [
    "scripts/notebook_tools/notebook_tools.py",
    "scripts/coordination/tapis_rapide.py",
]
print()
print("=== Classe 1 — CLI : invocation par subprocess ===")
for rel in cli_samples:
    tool = ROOT / rel
    r = subprocess.run([sys.executable, str(tool), "--help"],
                       capture_output=True, text=True,
                       encoding="utf-8", errors="replace",
                       timeout=40, env=UTF8_CHILD)
    first = r.stdout.strip().splitlines()
    print()
    print(f"[exit {r.returncode}] {rel} --help")
    print("   " + (first[0] if first else "(vide)"))

Racine du depot : D:\dev\CoursIA-agentinterfaces-02

=== Classe 1 — CLI : invocation par subprocess ===



[exit 0] scripts/notebook_tools/notebook_tools.py --help
   usage: notebook_tools.py [-h]

[exit 0] scripts/coordination/tapis_rapide.py --help
   usage: tapis_rapide.py [-h] --lane LANE [--grains GRAINS] [--json]


In [2]:
# --- Classe 2 : serveur MCP (le .mcp.json est le contrat) ---
# Un harnais (Claude Code, Roo, ...) lit .mcp.json et lance
#     <command> <args...>
# sur un pipe stdio. Le carnet demontre la ligne de demarrage REELLE et
# verifie que le binaire de la commande existe bien sur cette machine.
import json

cfg = json.loads((ROOT / ".mcp.json").read_text(encoding="utf-8"))
servers = cfg.get("mcpServers", {})
print()
print("=== Classe 2 — MCP : contrat stdio de .mcp.json ===")
print(f"Serveurs declares : {len(servers)}")
for name, spec in servers.items():
    cmd = spec.get("command", "")
    args = spec.get("args", [])
    launch = " ".join([cmd] + args)
    print()
    print(f"  {name}  (type: {spec.get('type', 'stdio')})")
    print(f"    ligne de demarrage : {launch}")
    print(f"    binaire present      : {Path(cmd).is_file()}  ({cmd})")


=== Classe 2 — MCP : contrat stdio de .mcp.json ===
Serveurs declares : 1

  jupyter-papermill  (type: stdio)
    ligne de demarrage : D:/dev/CoursIA/venv/Scripts/python.exe -m papermill_mcp.main
    binaire present      : True  (D:/dev/CoursIA/venv/Scripts/python.exe)


In [3]:
# --- Classe 3 : bridge notebook -> service HTTP ---
# Le bridge est un SERVICE : on le demarre avec --host/--port, puis un
# notebook l'interroge en HTTP. Le carnet demontre le contrat de demarrage
# (les flags reels du --help) sans demarrer le service.
r = subprocess.run(
    [sys.executable, str(ROOT / "scripts/notebook_tools/kernel_gateway_service.py"),
     "--help"],
    capture_output=True, text=True, encoding="utf-8", errors="replace",
    timeout=40, env=UTF8_CHILD,
)
print()
print("=== Classe 3 — Bridge : contrat HTTP (kernel_gateway_service.py) ===")
print(f"[exit {r.returncode}] --help")
flags = [ln.strip() for ln in r.stdout.splitlines()
         if ln.strip().startswith("--")]
print("  flags de demarrage (reels) :")
for fl in flags[:6]:
    print("   ", fl)


=== Classe 3 — Bridge : contrat HTTP (kernel_gateway_service.py) ===
[exit 0] --help
  flags de demarrage (reels) :
    --host HOST           adresse loopback (défaut : 127.0.0.1)
    --port PORT           port HTTP (défaut : 8899)
    --check-only          vérifie le seed et la dépendance, sans démarrer
    --probe-path PROBE_PATH
    --probe-json PROBE_JSON
    --probe-method {GET,POST}


## Lecture du résultat

Les trois contrats sont **réellement invocables** sur cette machine :

- **CLI** — `subprocess` lance l'outil, `argparse` répond à `--help` (exit 0). Un agent appelle n'importe lequel des 622 outils de `scripts/` de la même façon : `python <script> [args]`.
- **MCP** — `.mcp.json` donne la ligne de démarrage **stdio** exacte (`python -m papermill_mcp.main`) ; le binaire est **vérifié présent**. C'est exactement ce qu'un harnais exécute pour brancher le serveur.
- **Bridge** — `kernel_gateway_service.py` est un **service HTTP** (`--host` / `--port`) : c'est l'interface qu'un notebook interroge à distance, pas un simple script one-shot.

Le carnet 01 a dit **combien** (624 interfaces) ; celui-ci dit **comment** : chaque classe possède un contrat d'appel concret, testé.

## Exercices

**Exercice 1 — Appels CLI.** Écrire `appels_cli(liste)` qui, pour chaque chemin de `liste`, exécute `<script> --help` et renvoie la liste des codes de retour (un par script).

In [4]:
def appels_cli(liste):
    # TODO etudiant : pour chaque chemin,
    #   subprocess.run([sys.executable, chemin, "--help"], capture_output=True, ...)
    # puis retourner la liste des r.returncode
    return None

**Exercice 2 — Contrat MCP.** Écrire `ligne_mcp(nom)` qui lit `.mcp.json` et renvoie la chaîne de démarrage `<command> <args...>` du serveur `nom`.

In [5]:
def ligne_mcp(nom):
    # TODO etudiant : cfg = json.loads((ROOT / ".mcp.json").read_text(...))
    # spec = cfg["mcpServers"][nom]
    # retourner " ".join([spec["command"]] + spec["args"])
    return None

**Exercice 3 — Flags du bridge.** Écrire `flags_bridge()` qui exécute le `--help` de `kernel_gateway_service.py` et renvoie la liste de ses flags `--xxx`.

In [6]:
def flags_bridge():
    # TODO etudiant : executer --help de kernel_gateway_service.py,
    # retourner [l.strip() for l in stdout.splitlines() if l.strip().startswith("--")]
    return None

## Conclusion

Ce carnet ferme la boucle du carnet 01 : après avoir **mesuré** les interfaces (624), il **démontre** leurs trois contrats d'appel — CLI (`subprocess` + `--help`), MCP (`.mcp.json` stdio) et bridge (service HTTP) — par exécution réelle. Un agent qui lit ces deux carnets sait **quoi** utiliser et **comment** l'appeler.

See #27071.